<a href="https://colab.research.google.com/github/RomanCohort/notebooktmp/blob/main/RNAQwen_v3_Full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNAQwen v3 GitHub-only Colab

No Google Drive required. The notebook downloads:

- code and the two FSQ checkpoints from this GitHub repository;
- `rnaqa_v3_l234.jsonl` from the GitHub Release asset configured below;
- ERNIE-RNA and Qwen3.6-35B-A3B from their public model repositories.

ERNIE hidden states are streamed into a compact uint16 code cache. Checkpoints
can optionally be synced to a GitHub Release using a Colab secret named
`GITHUB_TOKEN`.

In [ ]:
GITHUB_REPO = 'RomanCohort/notebooktmp'
GITHUB_BRANCH = 'main'
DATASET_RELEASE_TAG = 'dataset-v3'
CHECKPOINT_RELEASE_TAG = 'train-v3-checkpoints'
FSQ_RELEASE_TAG = 'fsq-stream-v1'
CODES_RELEASE_TAG = 'codes-v3-cand'
DATASET_NAME = 'rnaqa_v3_cand.jsonl'
DATASET_SHA256 = ''  # 首次运行后回填
FSQ_SHA256 = 'e4cacbb5c0e64d321b59c619783dd6b2c408da8cd1888242dc5f152271c7feab'
CODE_EMB_SHA256 = '23504f2f942a07e769beb1872989fab74149afe1f3ebd07c267f6a7326fd2396'
CODES_SHA256 = ''  # 首次运行后回填
CODES_INDEX_SHA256 = ''  # 首次运行后回填
DATASET_URL = (
    f'https://github.com/{GITHUB_REPO}/releases/download/'
    f'{DATASET_RELEASE_TAG}/{DATASET_NAME}'
)
print('Code:', f'https://github.com/{GITHUB_REPO}')
print('Dataset:', DATASET_URL)

In [ ]:
import subprocess
subprocess.run([
    'nvidia-smi', '--query-gpu=name,memory.total,driver_version',
    '--format=csv,noheader'
], check=False)

In [ ]:
%pip install -q transformers==5.9.0 peft==0.21.2 accelerate==1.15.0 bitsandbytes==0.50.2 safetensors==0.8.0 sentencepiece==0.2.2 huggingface_hub==1.33.0 multimolecule==0.2.1 ninja
import hashlib, importlib, platform, subprocess, sys
import requests
def check_fla():
    fla = importlib.import_module('fla')
    ops = importlib.import_module('fla.ops.gated_delta_rule')
    if not hasattr(ops, 'chunk_gated_delta_rule'):
        raise ImportError('fla.ops.gated_delta_rule.chunk_gated_delta_rule missing')
    return fla

try:
    fla = check_fla()
except Exception:
    rc = subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'flash-linear-attention[cuda]==0.6.0',
    ], check=False).returncode
    importlib.invalidate_caches()
    fla = check_fla()
    print('FLA install return code:', rc)
print('Linear-attention fast path: FLA', getattr(fla, '__version__', 'installed'))

def causal_conv1d_wheel_url():
    import torch
    cuda = (torch.version.cuda or '').split('.')[0]
    torch_minor = '.'.join(torch.__version__.split('+', 1)[0].split('.')[:2])
    py = f'cp{sys.version_info.major}{sys.version_info.minor}'
    abi = 'TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
    arch = 'linux_x86_64' if platform.machine() in ('x86_64', 'amd64') else 'linux_aarch64'
    want = f'+cu{cuda}torch{torch_minor}cxx11abi{abi}'
    suffix = f'-{py}-{py}-{arch}.whl'
    signature = f'{py}/torch{torch_minor}/cu{cuda}/abi{abi}/{arch}'
    releases = [
        (GITHUB_REPO, 'kernel-wheels-v1'),
        ('Dao-AILab/causal-conv1d', 'v1.7.0'),
    ]
    for repo, tag in releases:
        api = f'https://api.github.com/repos/{repo}/releases/tags/{tag}'
        try:
            rel = requests.get(api, timeout=30).json()
        except Exception:
            continue
        for asset in rel.get('assets', []):
            name = asset['name']
            if want.lower() in name.lower() and name.endswith(suffix):
                expected = None
                sums = next((a for a in rel.get('assets', [])
                             if a['name'] == 'SHA256SUMS.txt'), None)
                if sums:
                    text = requests.get(sums['browser_download_url'], timeout=30).text
                    for line in text.splitlines():
                        parts = line.split()
                        if len(parts) >= 2 and parts[-1].split('/')[-1] == name:
                            expected = parts[0].lower()
                            break
                return asset['browser_download_url'], name, expected, signature
    return None, None, None, signature

try:
    causal_conv1d = importlib.import_module('causal_conv1d')
    print('causal-conv1d required kernel: installed')
except Exception:
    wheel_url, wheel_name, expected, signature = causal_conv1d_wheel_url()
    if wheel_url is None:
        raise RuntimeError(
            'causal-conv1d is required but no exact prebuilt wheel matches '
            f'{signature}')
    from pathlib import Path
    dst = Path('/tmp') / wheel_name
    part = dst.with_suffix(dst.suffix + '.part')
    with requests.get(wheel_url, stream=True, timeout=(30, 600)) as r:
        r.raise_for_status()
        with part.open('wb') as fh:
            for chunk in r.iter_content(8 * 1024 * 1024):
                if chunk:
                    fh.write(chunk)
    if expected:
        h = hashlib.sha256()
        with part.open('rb') as fh:
            for chunk in iter(lambda: fh.read(8 * 1024 * 1024), b''):
                h.update(chunk)
        if h.hexdigest().lower() != expected:
            raise RuntimeError(f'causal-conv1d wheel SHA256 mismatch: {wheel_name}')
    part.replace(dst)
    rc = subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', str(dst),
    ], check=False).returncode
    if rc != 0:
        raise RuntimeError(f'causal-conv1d wheel install failed: {rc} {dst}')
    importlib.invalidate_caches()
    importlib.import_module('causal_conv1d')
    print('causal-conv1d required kernel: wheel installed')

def run_pip_check():
    return subprocess.run(
        [sys.executable, '-m', 'pip', 'check'],
        text=True, capture_output=True, check=False,
    )

check = run_pip_check()
if 'ipython' in check.stdout.lower() and 'jedi' in check.stdout.lower():
    rc = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'jedi'],
        check=False,
    ).returncode
    if rc != 0:
        raise RuntimeError(f'Colab IPython repair failed: jedi install rc={rc}')
    check = run_pip_check()
print(check.stdout.strip())
relevant = ('transformers', 'torch', 'peft', 'accelerate', 'bitsandbytes',
            'flash', 'causal', 'multimolecule', 'huggingface', 'ipython')
bad = [line for line in check.stdout.splitlines()
       if any(name in line.lower() for name in relevant)]
if bad:
    raise RuntimeError('Dependency conflicts:\\n' + '\\n'.join(bad))
%pip cache purge

In [ ]:
from pathlib import Path
import hashlib, io, os, shutil, subprocess, sys, time, zipfile
import requests

ROOT = Path('/content/RNAQwen')
ASSET_DIR = Path('/content/rnaqwen_assets')
ASSET_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = ASSET_DIR / 'runs'
legacy_runs = ROOT / 'runs'
if legacy_runs.exists() and not RUNS_DIR.exists():
    shutil.move(str(legacy_runs), str(RUNS_DIR))
    print('Migrated runs to', RUNS_DIR)
elif legacy_runs.exists():
    backup_runs = ASSET_DIR / f'runs_legacy_{int(time.time())}'
    shutil.move(str(legacy_runs), str(backup_runs))
    print('Preserved conflicting legacy runs at', backup_runs)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
os.environ['RNAQ_RUNS_DIR'] = str(RUNS_DIR)
# Move the old in-repo data out before refreshing code.
legacy_data = ROOT / 'data'
asset_data = ASSET_DIR / 'data'
if legacy_data.exists() and not asset_data.exists():
    shutil.move(str(legacy_data), str(asset_data))
    print('Migrated data to', asset_data)
legacy_ernie = ROOT / 'models' / 'ernierna'
asset_ernie = ASSET_DIR / 'models' / 'ernierna'
if legacy_ernie.exists() and not asset_ernie.exists():
    asset_ernie.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(legacy_ernie), str(asset_ernie))
    print('Migrated ERNIE-RNA to', asset_ernie)

try:
    meta = requests.get(
        f'https://api.github.com/repos/{GITHUB_REPO}/commits/{GITHUB_BRANCH}',
        timeout=30,
    ).json()
    revision = meta['sha']
except Exception:
    revision = 'unknown'
marker = ROOT / '.code_revision'
reuse_code = (
    revision != 'unknown' and ROOT.exists() and marker.exists()
    and marker.read_text(encoding='utf-8').strip() == revision
    and (ROOT / 'scripts/50_routeC_e2e.py').exists()
)
if not reuse_code:
    if ROOT.exists():
        shutil.rmtree(ROOT)
    archive_url = f'https://codeload.github.com/{GITHUB_REPO}/zip/refs/heads/{GITHUB_BRANCH}'
    r = requests.get(archive_url, timeout=(30, 300))
    print('GitHub archive:', r.status_code, len(r.content), 'bytes')
    if not r.ok:
        raise RuntimeError(f'GitHub archive download failed: {r.status_code} {r.text[:500]}')
    tmp_root = Path('/content/_rnaqwen_repo')
    if tmp_root.exists():
        shutil.rmtree(tmp_root)
    tmp_root.mkdir(parents=True)
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        zf.extractall(tmp_root)
    inner = next(p for p in tmp_root.iterdir() if p.is_dir())
    inner.rename(ROOT)
    shutil.rmtree(tmp_root, ignore_errors=True)
    marker.write_text(revision, encoding='utf-8')
else:
    print('Code already ready at revision', revision)
ckpt_dir = ROOT / 'checkpoints' / 'fsq_stream'
ckpt_dir.mkdir(parents=True, exist_ok=True)
def asset_sha(path):
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(16 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

for name, expected, expected_sha in [('tokenizer.pt', 1_106_665, FSQ_SHA256),
                                     ('code_emb_contentinit.pt', 46_139_405, CODE_EMB_SHA256)]:
    dst = ckpt_dir / name
    if (dst.exists() and dst.stat().st_size == expected
            and asset_sha(dst) == expected_sha):
        continue
    url = (f'https://github.com/{GITHUB_REPO}/releases/download/'
           f'{FSQ_RELEASE_TAG}/{name}')
    rr = requests.get(url, timeout=(30, 300))
    print('FSQ asset:', name, rr.status_code, len(rr.content), 'bytes')
    if not rr.ok or len(rr.content) != expected:
        raise RuntimeError(f'FSQ download failed: {name} {rr.status_code} {len(rr.content)}')
    dst.write_bytes(rr.content)
    got_sha = asset_sha(dst)
    if got_sha != expected_sha:
        dst.unlink(missing_ok=True)
        raise RuntimeError(f'FSQ SHA256 mismatch: {name} {got_sha} != {expected_sha}')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Code ready:', ROOT)

In [ ]:
import requests, time
DATA_DIR = ASSET_DIR / 'data' / 'processed'
DATA_DIR.mkdir(parents=True, exist_ok=True)
dataset_path = DATA_DIR / DATASET_NAME
expected_size = 663_240_811

def sha256_file(path):
    import hashlib
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(16 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def download_resumable(url, dst, expected, expected_sha):
    part = dst.with_suffix(dst.suffix + '.part')
    if dst.exists() and dst.stat().st_size == expected:
        if sha256_file(dst) == expected_sha:
            print('Dataset already downloaded:', dst)
            return
        print('Dataset hash mismatch; downloading again:', dst)
        dst.unlink()
    have = part.stat().st_size if part.exists() else 0
    headers = {'Range': f'bytes={have}-'} if have else {}
    mode = 'ab' if have else 'wb'
    with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as r:
        if r.status_code not in (200, 206):
            raise RuntimeError(f'Dataset download failed: HTTP {r.status_code} {url}')
        if have and r.status_code == 200:
            have, mode = 0, 'wb'
        total = int(r.headers.get('Content-Length', 0)) + have
        with part.open(mode) as fh:
            for chunk in r.iter_content(8 * 1024 * 1024):
                if chunk:
                    fh.write(chunk)
                    have += len(chunk)
                    if have % (128 * 1024 * 1024) < len(chunk):
                        print(f'  {have / 2**20:.0f} / {total / 2**20:.0f} MiB', flush=True)
    if part.stat().st_size != expected:
        raise RuntimeError(f'Incomplete dataset: {part.stat().st_size} != {expected}')
    got_sha = sha256_file(part)
    if got_sha != expected_sha:
        part.unlink(missing_ok=True)
        raise RuntimeError(f'Dataset SHA256 mismatch: {got_sha} != {expected_sha}')
    os.replace(part, dst)

download_resumable(DATASET_URL, dataset_path, expected_size, DATASET_SHA256)

In [ ]:
from huggingface_hub import snapshot_download
ERNIE_DIR = ASSET_DIR / 'models' / 'ernierna'
ERNIE_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id='multimolecule/ernierna',
    local_dir=str(ERNIE_DIR),
)
print('ERNIE-RNA ready:', ERNIE_DIR)

In [ ]:
codes_path = DATA_DIR / '_rnaqa_v3_cand_codes.u16'
codes_index = DATA_DIR / '_rnaqa_v3_cand_codes_index.pt'

def download_asset(tag, name, dst):
    url = f'https://github.com/{GITHUB_REPO}/releases/download/{tag}/{name}'
    part = dst.with_suffix(dst.suffix + '.part')
    with requests.get(url, stream=True, timeout=(30, 300)) as r:
        r.raise_for_status()
        with part.open('wb') as fh:
            for chunk in r.iter_content(8 * 1024 * 1024):
                if chunk:
                    fh.write(chunk)
    os.replace(part, dst)

def validate_code_cache():
    """结构校验：不硬编码样本数，只验证自洽性和与数据集的对应关系。"""
    if not (codes_path.exists() and codes_index.exists()):
        return False
    import torch
    try:
        ci = torch.load(codes_index, map_location='cpu', weights_only=False)
    except Exception as exc:
        print('Code cache index is corrupt:', exc)
        return False
    n_pos = int(ci['n_pos'])
    checks = {
        'bytes': codes_path.stat().st_size == n_pos * 2,
        'vocab': int(ci['vocab']) == 4096,
        'pool': int(ci['pool']) == 4,
        'dataset': ci.get('dataset_sha256') == DATASET_SHA256,
        'fsq': ci.get('fsq_sha256') == FSQ_SHA256,
        'offset_start': int(ci['offsets'][0]) == 0,
        'offset_end': int(ci['offsets'][-1]) + int(ci['lens'][-1]) == n_pos,
        'contiguous': bool(torch.equal(
            ci['offsets'][1:], ci['offsets'][:-1] + ci['lens'][:-1])),
        'uid_count': len(ci['uids']) > 100_000,
    }
    print('Code cache validation:', checks, '| n_pos', n_pos, '| uids', len(ci['uids']))
    return all(checks.values())

if not validate_code_cache():
    for stale in (codes_path, codes_index):
        stale.unlink(missing_ok=True)
    print('Downloading published codes-v3-cand cache...')
    try:
        download_asset(CODES_RELEASE_TAG, codes_index.name, codes_index)
        download_asset(CODES_RELEASE_TAG, codes_path.name, codes_path)
    except Exception as exc:
        print('Download failed, encoding locally:', exc)
        for stale in (codes_path, codes_index):
            stale.unlink(missing_ok=True)
        free_gb = shutil.disk_usage('/content').free / 2**30
        if free_gb < 68.0:
            raise RuntimeError('Need at least 68 GiB free.')
        subprocess.run([
            sys.executable, '-X', 'utf8', '-u',
            str(ROOT / 'scripts/81_encode_v3_codes.py'),
            '--data', str(dataset_path), '--encoder', str(ERNIE_DIR),
            '--fsq', str(ROOT / 'checkpoints/fsq_stream/tokenizer.pt'),
            '--out', str(codes_path), '--idx', str(codes_index),
            '--pool', '4', '--batch', '16', '--sort-by-length',
        ], cwd=ROOT, check=True)
    if not validate_code_cache():
        raise RuntimeError('Code cache validation failed after download/encoding')
print('Code cache ready:', codes_path)

In [ ]:
import json

MODEL_DIR = Path('/content/models/Qwen3.6-35B-A3B')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_REPO = 'Qwen/Qwen3.6-35B-A3B'
index_path = MODEL_DIR / 'model.safetensors.index.json'

def _missing_shards():
    """Return the shard names the index declares but the directory lacks.

    An empty list means the model is complete. The previous version gated on
    config.json, which a partial ModelScope run leaves behind: the download was
    then skipped and the index check blew up afterwards.
    """
    if not index_path.exists():
        return None
    try:
        idx = json.loads(index_path.read_text(encoding='utf-8'))
        names = set(idx['weight_map'].values())
    except Exception as exc:
        print('Unreadable index:', exc, flush=True)
        return None
    return sorted(n for n in names if not (MODEL_DIR / n).exists())

def _hf_download():
    """HuggingFace first: Colab runs in the US and ModelScope is hosted in
    China, so the 11 MB/s vs 0.06 MB/s measurement in 00b_*.py (taken from a
    China-based host) is inverted here. hf_transfer is a Rust multi-threaded
    downloader and is several times faster than the default Python path."""
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    try:
        import hf_transfer  # noqa: F401
    except Exception:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'hf_transfer'], check=True)
        import hf_transfer  # noqa: F401
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=MODEL_REPO, local_dir=str(MODEL_DIR),
                      max_workers=8)

def _ms_download():
    subprocess.run([
        sys.executable, '-u', str(ROOT / 'scripts/00b_download_modelscope.py'),
        '--repo', MODEL_REPO,
        '--dest', str(MODEL_DIR),
        '--workers', '4'
    ], cwd=ROOT, check=True)

missing = _missing_shards()
have = sorted(p.name for p in MODEL_DIR.iterdir())[:6] if MODEL_DIR.exists() else []
print(f'Model dir currently holds {len(list(MODEL_DIR.iterdir()))} entries: {have}', flush=True)

if missing is None or missing:
    if index_path.exists():
        print(f'Index present but {len(missing)} shards missing; repairing.', flush=True)
    else:
        print('No index yet; downloading the full model.', flush=True)
    try:
        print('Downloading from HuggingFace with hf_transfer ...', flush=True)
        _hf_download()
    except Exception as exc:
        print('HuggingFace download failed:', exc, flush=True)
        print('Falling back to ModelScope ...', flush=True)
        _ms_download()

missing = _missing_shards()
if missing is None:
    raise FileNotFoundError(
        f'{index_path} still missing after download. Directory contains '
        f'{len(list(MODEL_DIR.iterdir()))} entries: '
        f'{sorted(p.name for p in MODEL_DIR.iterdir())[:10]}')
if missing:
    raise RuntimeError(
        f'Incomplete model, {len(missing)} shards missing: {missing[:3]}')

os.environ['RNAQ_QWEN36_DIR'] = str(MODEL_DIR)
os.environ['RNAQ_THREADS'] = '8'
print('Model ready:', MODEL_DIR)


## A100 40GB memory budget

| item | GiB |
|---|---|
| Qwen3.6-35B-A3B int4 weights | 18.2 |
| SOS/LoRA trainable params | 0.1 |
| measured reserved, B*T = 4096 with checkpointing | 23.4 |
| estimated reserved, B*T = 8192 with checkpointing | 28-33 |
| estimated A100 40GB headroom at B*T = 8192 | 7-12 |

`--token-budget 512` was an APU workaround. On a 40GB A100 the binding
constraint is activation memory, not the weights, so the notebook uses
`--token-budget 8192 --batch 64 --grad-ckpt 1`.

Qwen3.6 has 30 linear-attention layers. The setup cell installs
`flash-linear-attention[cuda]` and `causal-conv1d`; without them Transformers
falls back to the slow pure-PyTorch chunk/recurrent implementation. The first
three training steps also print a `[profile]` timing split. The exact
Python 3.13 / torch 2.11 / CUDA 13 wheel is published at
`kernel-wheels-v1` and checked by SHA256 before installation.

Step count is `11625`, preserving the previous total token budget after
doubling tokens per step. This is **not one epoch**: the
training split contains 831,390 QA rows, and token packing fragments batches.
The sampler advances without replacement within each epoch, so repeated rows no
longer crowd out unseen data. `--min-steps 5000` keeps early stopping from
firing before the run has seen a meaningful fraction of the data.

This notebook starts full training directly. The first step includes
cold-start kernel initialization; compare the profile from steps 2-3. GPU
utilization is sampled every 0.75 seconds and reported as a three-second window
average and peak, with a min/median/mean/max summary at segment exit.


In [ ]:
common = [
    sys.executable, '-X', 'utf8', '-u', str(ROOT / 'scripts/50_routeC_e2e.py'),
    '--data', str(dataset_path),
    '--codes', str(codes_path),
    '--codes-idx', str(codes_index),
    '--fsq', str(ROOT / 'checkpoints/fsq_stream/tokenizer.pt'),
    '--init-code-emb', str(ROOT / 'checkpoints/fsq_stream/code_emb_contentinit.pt'),
    '--base', 'qwen36-35b', '--qwen36-mode', 'int4',
    '--adapter', 'sos', '--lora-r', '8',
    # 候选式任务：序列 + 4 选 1，答案只有一个字母，序列长度是主要成本。
    # token-budget 8192 / batch 64 沿用 A100 40GB 的既有配置。
    '--token-budget', '8192', '--batch', '64', '--bucket', '256',
    '--experts', 'grouped',
    '--profile-steps', '3',
    '--grad-ckpt', '1',
]
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
print('Candidate-discrimination training configured.')

In [ ]:
from google.colab import userdata
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
print('Checkpoint sync enabled:', bool(GITHUB_TOKEN))

API = 'https://api.github.com'
HEADERS = {'Accept': 'application/vnd.github+json'}
if GITHUB_TOKEN:
    HEADERS['Authorization'] = f'Bearer {GITHUB_TOKEN}'

CKPT_DIR = RUNS_DIR / 'e2e_v3_cand' / 'ckpt'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def release_info(create=False):
    url = f'{API}/repos/{GITHUB_REPO}/releases/tags/{CHECKPOINT_RELEASE_TAG}'
    r = requests.get(url, headers=HEADERS, timeout=30)
    if r.status_code == 200:
        return r.json()
    if not create or not GITHUB_TOKEN:
        return None
    r = requests.post(
        f'{API}/repos/{GITHUB_REPO}/releases', headers=HEADERS, timeout=30,
        json={'tag_name': CHECKPOINT_RELEASE_TAG,
              'name': 'RNAQwen v3 checkpoints',
              'body': 'Latest resumable training checkpoint.'},
    )
    if r.status_code not in (200, 201):
        raise RuntimeError(f'Create release failed: {r.status_code} {r.text[:300]}')
    return r.json()

def remote_assets():
    rel = release_info(False)
    return list(rel.get('assets', [])) if rel else []

def checkpoint_rank(name):
    if name == 'final.pt':
        return (2, 0)
    if name == 'best.pt':
        return (1, 0)
    if name.startswith('step_') and name.endswith('.pt'):
        try:
            return (0, int(name[5:-3]))
        except ValueError:
            pass
    return (-1, -1)

def remote_checkpoints():
    return sorted(
        [a for a in remote_assets() if checkpoint_rank(a['name'])[0] >= 0],
        key=lambda a: a.get('updated_at', ''),
        reverse=True,
    )

def remote_timestamp(asset):
    from datetime import datetime
    return datetime.fromisoformat(
        asset.get('updated_at', '1970-01-01T00:00:00Z').replace('Z', '+00:00')
    ).timestamp()

def valid_checkpoint(path):
    try:
        with zipfile.ZipFile(path) as zf:
            return bool(zf.namelist())
    except Exception:
        return False

def local_checkpoints():
    candidates = list(CKPT_DIR.glob('step_*.pt'))
    candidates += [CKPT_DIR / n for n in ('best.pt', 'final.pt') if (CKPT_DIR / n).exists()]
    good = []
    for p in candidates:
        if valid_checkpoint(p):
            good.append(p)
        else:
            print('Discarding corrupt local checkpoint:', p, flush=True)
            p.unlink(missing_ok=True)
    return sorted(good, key=lambda p: p.stat().st_mtime, reverse=True)

def download_checkpoint(asset):
    dst = CKPT_DIR / asset['name']
    part = dst.with_suffix(dst.suffix + '.part')
    with requests.get(asset['url'], headers={**HEADERS, 'Accept': 'application/octet-stream'},
                      stream=True, timeout=(30, 600)) as r:
        r.raise_for_status()
        with part.open('wb') as fh:
            for chunk in r.iter_content(8 * 1024 * 1024):
                if chunk:
                    fh.write(chunk)
            fh.flush()
            os.fsync(fh.fileno())
    if not valid_checkpoint(part):
        part.unlink(missing_ok=True)
        raise RuntimeError(f'Downloaded checkpoint is corrupt: {asset["name"]}')
    os.replace(part, dst)
    return dst

def ensure_best_checkpoint():
    best = CKPT_DIR / 'best.pt'
    if best.exists() and valid_checkpoint(best):
        return best
    asset = next((a for a in remote_assets() if a['name'] == 'best.pt'), None)
    if not asset:
        return None
    try:
        return download_checkpoint(asset)
    except Exception as exc:
        print('best.pt restore failed:', exc, flush=True)
        return None

def restore_latest_checkpoint():
    local = local_checkpoints()
    remote = remote_checkpoints()
    if local:
        local_names = {p.name for p in local}
        if (remote and remote[0]['name'] not in local_names
                and remote_timestamp(remote[0]) > local[0].stat().st_mtime):
            try:
                dst = download_checkpoint(remote[0])
                ensure_best_checkpoint()
                return dst
            except Exception as exc:
                print('Remote checkpoint failed; keeping local:', exc, flush=True)
        ensure_best_checkpoint()
        return local[0]
    for asset in remote:
        try:
            dst = download_checkpoint(asset)
            ensure_best_checkpoint()
            print('Restored remote checkpoint:', dst)
            return dst
        except Exception as exc:
            print('Remote checkpoint rejected:', asset['name'], exc, flush=True)
    return None

def upload_checkpoint(path, cleanup_steps=False):
    if not GITHUB_TOKEN:
        return
    rel = release_info(True)
    assets = remote_assets()
    existing = next((a for a in assets if a['name'] == path.name), None)
    if existing and existing.get('size') == path.stat().st_size:
        print('Already uploaded:', path.name)
    else:
        if existing:
            requests.delete(existing['url'], headers=HEADERS, timeout=30)
        url = rel['upload_url'].split('{', 1)[0]
        with path.open('rb') as fh:
            r = requests.post(
                url, headers={**HEADERS, 'Content-Type': 'application/octet-stream'},
                params={'name': path.name}, data=fh, timeout=(30, 900),
            )
        if r.status_code not in (200, 201):
            raise RuntimeError(f'Upload failed: {r.status_code} {r.text[:300]}')
        print('Uploaded:', path.name)
    if cleanup_steps:
        for old in remote_assets():
            if old['name'].startswith('step_') and old['name'] != path.name:
                requests.delete(old['url'], headers=HEADERS, timeout=30)

In [ ]:
import csv, queue, shlex, statistics, threading, time
from IPython.display import HTML, display
restored = restore_latest_checkpoint()
full = common + [
    # 候选式数据集 train 87,307 条；真实步长 p50=156（比原数据集长 40%，
    # 因为 question 里带 4 个候选），token-budget 8192 -> 约 52 条/步。
    # 1 epoch 约 1,679 步；8,000 步约 4.8 epoch。
    '--steps', '8000', '--min-steps', '2000',
    '--early-stop', '1', '--eval-every', '250',
    '--eval-n', '64',
    '--log-every', '1',
    '--patience', '6', '--min-delta', '0.005',
    '--diag', '0',
    '--save-every', '50', '--save-minutes', '10', '--keep-ckpt', '3',
    '--tag', 'v3_cand'
]
if restored:
    full += ['--resume', str(restored)]
    print('Resuming from:', restored)
else:
    print('Starting a new full run.')

stop_sync = threading.Event()
def sync_loop():
    seen = set()
    while not stop_sync.is_set():
        local = sorted(CKPT_DIR.glob('step_*.pt'))
        if local:
            p = local[-1]
            key = (p.name, p.stat().st_size, int(p.stat().st_mtime))
            if key not in seen:
                try:
                    upload_checkpoint(p, cleanup_steps=True)
                    seen.add(key)
                except Exception as exc:
                    print('Checkpoint sync error:', exc, flush=True)
        stop_sync.wait(60)

thread = threading.Thread(target=sync_loop, daemon=True)
thread.start()

gpu_metrics_path = RUNS_DIR / 'e2e_v3_cand' / 'gpu_metrics.csv'
gpu_metrics_path.parent.mkdir(parents=True, exist_ok=True)
gpu_header = ('timestamp,gpu_util_pct,gpu_mem_util_pct,vram_used_mib,'
              'vram_total_mib,temperature_c,power_w,sm_clock_mhz,mem_clock_mhz,pstate\\n')
if (not gpu_metrics_path.exists() or gpu_metrics_path.stat().st_size == 0
        or gpu_metrics_path.read_text(encoding='utf-8').splitlines()[0] != gpu_header.strip()):
    gpu_metrics_path.write_text(gpu_header, encoding='utf-8')
gpu_panel = display(HTML('<b>GPU monitor starting...</b>'), display_id=True)
GPU_SAMPLE_SECONDS = 0.75
GPU_REPORT_SECONDS = 3.0
train_log = RUNS_DIR / 'e2e_v3_cand' / 'train.log'

quoted = ' '.join(shlex.quote(x) for x in full)
shell_cmd = 'set -o pipefail; timeout --signal=TERM 27000 ' + quoted
print(shell_cmd)
proc = subprocess.Popen(
    ['bash', '-lc', shell_cmd], cwd=ROOT,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
line_queue = queue.Queue()
def read_lines():
    for line in proc.stdout:
        line_queue.put(line)
    line_queue.put(None)
reader = threading.Thread(target=read_lines, daemon=True)
reader.start()

rc = -1
last_gpu = 0.0
last_gpu_report = 0.0
gpu_window = []
gpu_util_history = []
gpu_peak_history = []
query = ('timestamp,utilization.gpu,utilization.memory,memory.used,memory.total,'
         'temperature.gpu,power.draw,clocks.current.sm,clocks.current.memory,pstate')

def metric_num(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0
try:
    with train_log.open('a', encoding='utf-8') as train_fh:
        while True:
            try:
                out_line = line_queue.get(timeout=0.5)
            except queue.Empty:
                out_line = ''
            if out_line is None:
                break
            if out_line:
                print(out_line, end='', flush=True)
                train_fh.write(out_line)
                train_fh.flush()

            now = time.monotonic()
            if now - last_gpu >= GPU_SAMPLE_SECONDS:
                last_gpu = now
                try:
                    gpu_line = subprocess.run(
                        ['nvidia-smi', f'--query-gpu={query}',
                         '--format=csv,noheader,nounits'],
                        check=True, text=True, capture_output=True, timeout=10,
                    ).stdout.strip().splitlines()[0]
                    parts = [x.strip() for x in gpu_line.split(',')]
                    ts, util, mem_util, used, total, temp, power, sm_clk, mem_clk, pstate = parts[:10]
                    with gpu_metrics_path.open('a', encoding='utf-8', newline='') as fh:
                        csv.writer(fh).writerow(parts[:10])

                    util_value = min(100.0, max(0.0, metric_num(util)))
                    mem_util_value = metric_num(mem_util)
                    power_value = metric_num(power)
                    gpu_window.append((util_value, mem_util_value, power_value))

                    if now - last_gpu_report >= GPU_REPORT_SECONDS or proc.poll() is not None:
                        last_gpu_report = now
                        util_vals = [x[0] for x in gpu_window]
                        mem_vals = [x[1] for x in gpu_window]
                        power_vals = [x[2] for x in gpu_window]
                        avg_util = statistics.mean(util_vals)
                        peak_util = max(util_vals)
                        avg_mem_util = statistics.mean(mem_vals)
                        peak_power = max(power_vals)
                        active_pct = 100.0 * sum(x >= 5.0 for x in util_vals) / len(util_vals)
                        gpu_util_history.append(avg_util)
                        gpu_peak_history.append(peak_util)
                        vram_pct = 100.0 * float(used) / max(float(total), 1.0)
                        alive = proc.poll() is None
                        gpu_text = (
                            f'3s avg {avg_util:5.1f}%  peak {peak_util:5.1f}%  '
                            f'active {active_pct:5.1f}%  mem-avg {avg_mem_util:3.0f}%  '
                            f'VRAM {float(used):.0f}/{float(total):.0f} MiB '
                            f'({vram_pct:.1f}%)  temp {temp} C  power peak {peak_power:.1f} W  '
                            f'SM {sm_clk} MHz  mem {mem_clk} MHz  {pstate}  '
                            f'proc {"alive" if alive else "exited"}'
                        )
                        print(f'  [gpu] {gpu_text}', flush=True)
                        train_fh.write(f'  [gpu] {gpu_text}\n')
                        train_fh.flush()
                        gpu_panel.update(HTML(
                            '<div style="font-family:monospace;line-height:1.5">'
                            f'<b>GPU {ts} - last {GPU_REPORT_SECONDS:.0f}s</b><br>'
                            f'avg {avg_util:5.1f}%  peak {peak_util:5.1f}%  '
                            f'active {active_pct:5.1f}%<br>'
                            f'<progress value="{avg_util:.1f}" max="100"></progress><br>'
                            f'mem-util avg {avg_mem_util:.0f}%<br>'
                            f'VRAM {float(used):.0f}/{float(total):.0f} MiB ({vram_pct:.1f}%)  '
                            f'<progress value="{vram_pct:.1f}" max="100"></progress><br>'
                            f'temp {temp} C   power peak {peak_power:.1f} W<br>'
                            f'SM {sm_clk} MHz   mem {mem_clk} MHz   {pstate}<br>'
                            f'process {"alive" if alive else "exited"}</div>'
                        ))
                        gpu_window.clear()
                except Exception as exc:
                    print(f'  [gpu] unavailable: {exc}', flush=True)
    rc = proc.wait()
finally:
    stop_sync.set()
    if proc.poll() is None:
        proc.terminate()
    reader.join(timeout=15)
    thread.join(timeout=15)
if gpu_util_history:
    print(
        f'[gpu summary] windows {len(gpu_util_history)}  '
        f'avg-min {min(gpu_util_history):.1f}%  '
        f'avg-median {statistics.median(gpu_util_history):.1f}%  '
        f'avg-mean {statistics.mean(gpu_util_history):.1f}%  '
        f'avg-max {max(gpu_util_history):.1f}%  '
        f'instant-peak {max(gpu_peak_history):.1f}%',
        flush=True,
    )
local = local_checkpoints()
steps = [p for p in local if p.name.startswith('step_')]
if steps:
    upload_checkpoint(steps[0], cleanup_steps=True)
for name in ('best.pt', 'final.pt'):
    p = CKPT_DIR / name
    if p.exists() and valid_checkpoint(p):
        upload_checkpoint(p)
artifacts = RUNS_DIR / 'e2e_v3_cand' / 'artifacts'
if artifacts.exists():
    artifact_zip = Path(shutil.make_archive(
        str(RUNS_DIR / 'e2e_v3_cand' / 'artifacts'), 'zip', root_dir=artifacts))
    upload_checkpoint(artifact_zip)
summary = RUNS_DIR / 'e2e_v3_cand' / 'summary.json'
if summary.exists():
    upload_checkpoint(summary)
print('Full-run segment exit code:', rc)

In [ ]:
# ---- 候选式准确率评测：4 选 1，随机基线 25% ----
import json, random, torch
from pathlib import Path

def eval_candidate_accuracy(ckpt_path, split='test', n=200, seed=7):
    """从 checkpoint 载模型，在候选式样本上算 4 选 1 准确率。"""
    import importlib.util
    sys.path.insert(0, str(ROOT / 'src'))
    import rnaqwen  # noqa: F401  先装 shim
    spec = importlib.util.spec_from_file_location(
        'rc', ROOT / 'scripts/50_routeC_e2e.py')
    rc = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(rc)

    ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cargs = ck.get('args', {})
    from rnaqwen.qwen36 import base_spec_from_ckpt_args
    base_dir, mk_kw = base_spec_from_ckpt_args(cargs)

    import numpy as np
    dc = torch.load(codes_index, map_location='cpu', weights_only=False)
    mm = np.memmap(codes_path, dtype=np.uint16, mode='r', shape=(int(dc['n_pos']),))
    CODES = torch.from_numpy(mm)
    idx = {u: (int(dc['offsets'][i]), int(dc['lens'][i]))
           for i, u in enumerate(dc['uids'])}
    LAT, LEV = int(dc['latent']), int(dc['levels'])

    from transformers import AutoTokenizer
    tk0 = AutoTokenizer.from_pretrained(str(base_dir))
    tk = rc.Tok(tk0)
    model = rc.E2EModel(int(cargs.get('lora_r', 8)), cargs.get('rna_emb', 'table'),
                        latent=LAT, levels=LEV, **mk_kw)
    tk.base_vocab = model.base_vocab
    model.fsq.load_state_dict(ck['fsq'])
    if ck.get('rna_emb') is not None:
        model.rna_emb.weight.data.copy_(
            ck['rna_emb'].to(model.rna_emb.weight.device, model.rna_emb.weight.dtype))
    model.llm.load_state_dict(ck['lora'], strict=False)
    model.eval()
    DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

    rows = []
    with open(dataset_path, encoding='utf-8') as f:
        for line in f:
            r = json.loads(line)
            if r.get('split') == split and r.get('kind') == 'candidate_family':
                rows.append(r)
    rng = random.Random(seed)
    rows = rng.sample(rows, min(n, len(rows)))

    hit = 0
    with torch.no_grad():
        for r in rows:
            off, ln = idx[r['uid']]
            code = CODES[off:off + ln].to(DEV, dtype=torch.long)
            v, _, _, _ = model.rna_vectors_from_codes(code)
            ids = (tk.sys_ids + tk.user_ids + tk.enc('Q: <RNA>')
                   + [tk.base_vocab] * v.shape[0]
                   + tk.enc(f"</RNA>{r['question']}")
                   + tk.assistant_ids + tk.no_think + tk.enc(' A:'))
            ii = torch.tensor([ids], device=DEV)
            emb = model.embed(ii, v)
            out = model.llm.generate(inputs_embeds=emb,
                                     attention_mask=torch.ones_like(ii),
                                     max_new_tokens=4, do_sample=False,
                                     pad_token_id=tk.pad_id)
            txt = tk0.decode(out[0], skip_special_tokens=True).strip()
            pred = txt.split('A:')[-1].strip()[:1].upper()
            if pred == r['answer']:
                hit += 1
    acc = hit / max(len(rows), 1)
    print(f'[{split}] 4 选 1 准确率 {100*acc:.1f}%  ({hit}/{len(rows)})  '
          f'随机基线 25.0%')
    return acc

ckpts = sorted(CKPT_DIR.glob('step_*.pt'), key=lambda p: int(p.stem.split('_')[1]))
if ckpts:
    eval_candidate_accuracy(ckpts[-1])
else:
    print('还没有 checkpoint，先跑训练')

## Resume after a disconnect

Run the setup cells again, then run the full-training cell. Checkpoints live
under `/content/rnaqwen_assets/runs`, survive a code refresh in the same runtime,
and are restored atomically from local storage or the GitHub Release.

Without `GITHUB_TOKEN` (or another persistent mount), checkpoints still vanish
when Colab destroys the runtime. Add a fine-grained GitHub token with Contents
read/write permission as `GITHUB_TOKEN` for cross-session resume.